# Abaqus CSV Plotting

Interactive notebook for local post-processing.

Workflow:
1. Set `input_dir` and `output_dir`
2. Run all cells
3. Adjust plotting code as needed for each case

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib import font_manager as fm

plt.style.use('default')

In [ ]:
# Update these paths per case
input_dir = Path(r"D:\develop\py\abaqus2py\data\Job-2")
output_dir = Path(r"D:\develop\py\abaqus2py\figures\Job-2")
plane = "xy"  # choose from: xy, xz, yz

# Figure export settings
show_dpi = 180
save_dpi = 300
save_figures = True
save_vector = True
save_png = True

# Figure geometry
curve_figsize = (4.4, 3.4)
stress_figsize = (6.6, 5.0)

title_text = ""
x_label = "Δ(mm)"
y_label = "N(kN)"

output_dir.mkdir(parents=True, exist_ok=True)
input_dir, output_dir

In [ ]:
def pick_font(candidates):
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            return name
    return None

english_font = pick_font(["Times New Roman", "Times New Roman PS MT", "DejaVu Serif"])
chinese_font = pick_font(["SimSun", "Songti SC", "STSong", "Noto Serif CJK SC"])

plt.rcParams["figure.dpi"] = show_dpi
plt.rcParams["savefig.dpi"] = save_dpi
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [x for x in [english_font, "Times New Roman", "DejaVu Serif"] if x]
plt.rcParams["axes.unicode_minus"] = False

english_font, chinese_font

In [ ]:
def save_plot(fig, stem):
    if not save_figures:
        return
    if save_png:
        fig.savefig(output_dir / f"{stem}.png", dpi=save_dpi, bbox_inches="tight", facecolor="white")
    if save_vector:
        fig.savefig(output_dir / f"{stem}.pdf", bbox_inches="tight", facecolor="white")
        fig.savefig(output_dir / f"{stem}.svg", bbox_inches="tight", facecolor="white")

In [ ]:
ld_csv = input_dir / "load_displacement.csv"
stress_csv = input_dir / "stress_cloud.csv"

ld_df = pd.read_csv(ld_csv)
stress_df = pd.read_csv(stress_csv)

print(ld_csv)
print(stress_csv)
display(ld_df.head())
display(stress_df.head())

In [ ]:
# Auto-detect displacement and reaction columns
x_candidates = [c for c in ld_df.columns if c.startswith("avg_U")]
y_candidates = [c for c in ld_df.columns if c.startswith("sum_")]

if not x_candidates or not y_candidates:
    raise ValueError("load_displacement.csv missing avg_U*/sum_* columns")

x_col = x_candidates[0]
y_col = y_candidates[0]
x_col, y_col

In [ ]:
label_font = {"fontsize": 12}
title_font = {"fontsize": 12}
if chinese_font:
    label_font["fontname"] = chinese_font
    title_font["fontname"] = chinese_font

tick_font = english_font or "DejaVu Serif"

fig, ax = plt.subplots(figsize=curve_figsize)

ax.plot(ld_df[x_col], ld_df[y_col], color="#5f5f5f", lw=1.6, solid_capstyle="round")

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
    spine.set_color("#6f6f6f")

ax.yaxis.grid(True, linestyle="-", linewidth=0.5, color="#d8d8d8", alpha=0.7)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

ax.tick_params(axis="both", which="major", direction="out", length=3.5, width=0.7, labelsize=10, color="#6f6f6f")
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontname(tick_font)

ax.set_xlabel(x_label, **label_font)
ax.set_ylabel(y_label, **label_font)
if title_text:
    ax.set_title(title_text, pad=8, **title_font)

fig.tight_layout()
save_plot(fig, "load_displacement")
plt.show()

In [ ]:
plane_map = {
    "xy": ("x", "y"),
    "xz": ("x", "z"),
    "yz": ("y", "z"),
}

c1, c2 = plane_map[plane]
triang = mtri.Triangulation(stress_df[c1].values, stress_df[c2].values)

In [ ]:
fig, ax = plt.subplots(figsize=stress_figsize)
cntr = ax.tricontourf(triang, stress_df["mises"].values, levels=24, cmap="turbo")
cbar = fig.colorbar(cntr, ax=ax, label="Mises Stress")
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel(c1, fontsize=11, fontname=english_font or "DejaVu Serif")
ax.set_ylabel(c2, fontsize=11, fontname=english_font or "DejaVu Serif")
ax.set_title(f"Stress Cloud ({plane.upper()} plane)", fontsize=12, fontname=english_font or "DejaVu Serif")
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()
save_plot(fig, f"stress_cloud_{plane}")
plt.show()

In [ ]:
# Optional: quick scatter view for debugging point density
fig, ax = plt.subplots(figsize=stress_figsize)
sc = ax.scatter(stress_df[c1], stress_df[c2], c=stress_df["mises"], s=8, cmap="turbo")
fig.colorbar(sc, ax=ax, label="Mises Stress")
ax.set_xlabel(c1, fontsize=11, fontname=english_font or "DejaVu Serif")
ax.set_ylabel(c2, fontsize=11, fontname=english_font or "DejaVu Serif")
ax.set_title(f"Stress Scatter ({plane.upper()} plane)", fontsize=12, fontname=english_font or "DejaVu Serif")
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()
plt.show()